## Data Cleaning: Rental Listings

Prepares the Kaggle London rental listings for the prediction model in RQ1.
Input: `data/raw/rent_ads_rightmove_extended.csv` (3,478 listings).
Output: `data/processed/listings_clean.csv`.

Each step follows the preprocessing specification in Section 3.4.1 of the
methodology. Exploratory profiling of this dataset is in `01_data_inventory.ipynb`.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/rent_ads_rightmove_extended.csv')
print(df.shape)

(3478, 13)


In [2]:
print(df.columns.tolist())

['address', 'subdistrict_code', 'rent', 'deposit', 'Let type', 'Furnish Type', 'Coucil Tax', 'PROPERTY TYPE', 'BEDROOMS', 'BATHROOMS', 'SIZE', 'avg_distance_to_nearest_station', 'nearest_station_count']


In [3]:
df.head()

,address,subdistrict_code,rent,deposit,Let type,Furnish Type,Coucil Tax,PROPERTY TYPE,BEDROOMS,BATHROOMS,SIZE,avg_distance_to_nearest_station,nearest_station_count
0,"Morse Close, Harefield, UB9",UB9,1200,1384,Long term,Unfurnished,Ask agent,Apartment,1.0,1.0,Ask agent,2.1,3
1,"Dolphin Square, London, SW1V",SW1V,1983,2288,Long term,"Furnished or unfurnished, landlord is flexible",Ask agent,Apartment,1.0,1.0,335 sq ft,0.4,3
2,"Warlters Road, Holloway, London, N7",N7,1800,0,Long term,Unfurnished,Ask agent,Apartment,1.0,1.0,Ask agent,0.5,3
3,"Rockingham Road, Uxbridge",NaN,1650,1903,Long term,Unfurnished,Ask agent,Apartment,2.0,1.0,Ask agent,1.6,3
4,"Northwick Avenue, Harrow, HA3",HA3,2700,3115,Long term,Unfurnished,Ask agent,End of Terrace,3.0,2.0,Ask agent,0.4,3


### Removing unusable columns

Three columns are removed. Council tax records the placeholder "Ask agent" in
3,446 of 3,478 rows, and floor area in 2,556. Neither holds enough real values
to be informative. The nearest-station count is near-constant at 3 in 3,469
rows, providing no variance for a model to learn from. The related distance
measure is retained, since it varies and transport access is a known rent driver.

In [4]:
print("Coucil Tax 'Ask agent' :", (df['Coucil Tax'] == 'Ask agent').sum(), "of", len(df))
print("Size  'Ask agent' :", (df['SIZE'] == 'Ask agent').sum(), "of", len(df))
print("nearest_station_count = 3 :", (df['nearest_station_count'] == 3).sum(), "of", len(df))

df = df.drop(columns = ['Coucil Tax', 'SIZE', 'nearest_station_count'])
print("Shape after drop : ", df.shape)

Coucil Tax 'Ask agent' : 3446 of 3478
Size  'Ask agent' : 2556 of 3478
nearest_station_count = 3 : 3469 of 3478
Shape after drop :  (3478, 10)


### Removing non-dwellings

Parking spaces (83) and garages (36) are advertised on the same platform but
are not residential lettings. Retaining them would mean training a single model
across two unrelated markets.

In [5]:
non_dwellings = ['Parking', 'Garages']

print(df['PROPERTY TYPE'].value_counts().loc[non_dwellings])

df = df[~df['PROPERTY TYPE'].isin(non_dwellings)]
print("Shape after droping non-dwellings :", df.shape)

PROPERTY TYPE
Parking    83
Garages    36
Name: count, dtype: int64
Shape after droping non-dwellings : (3359, 10)


### Consolidating property type

"Flat" and "Apartment" denote the same dwelling type in UK usage; the split
reflects listing agents' wording rather than a real distinction. A model treats
them as unrelated categories, so merging them gives 2,295 observations of one
class rather than splitting the evidence across two.

In [6]:
print("Before :", df['PROPERTY TYPE'].value_counts().loc[['Flat', 'Apartment']].to_dict())

df['PROPERTY TYPE'] = df['PROPERTY TYPE'].replace('Apartment', 'Flat')

print("After : Flat =", (df['PROPERTY TYPE'] == 'Flat').sum())

Before : {'Flat': 1260, 'Apartment': 1035}
After : Flat = 2295


### Checking bedroom coverage

Before imputing, the missing values are examined by property type to confirm
that the rule in Section 3.4.1 applies to them.

In [7]:
print("Bedrooms missing :", df['BEDROOMS'].isna().sum())
print()
print("Missing by property type:")
print(df[df['BEDROOMS'].isna()]['PROPERTY TYPE'].value_counts())

Bedrooms missing : 519

Missing by property type:
PROPERTY TYPE
Studio                          258
Flat                            128
House Share                      95
Flat Share                        8
Terraced                          4
Semi-Detached                     2
House of Multiple Occupation      2
Ground Flat                       1
Detached                          1
House                             1
Land                              1
Penthouse                         1
Name: count, dtype: int64


### Bedroom imputation

519 listings have no bedroom count. For 363 the value is implied by property
type: studios have no separate bedroom, and shared accommodation lets a single
room. The remaining 156, mostly flats, are genuinely unknown. These are dropped
rather than imputed, since bedrooms is a primary predictor and estimated values
would be learned by the model as if observed.

In [8]:
# Studios have no separate bedroom; a missing value means zero, not unknown
df.loc[df['PROPERTY TYPE'] == 'Studio', 'BEDROOMS'] = \
    df.loc[df['PROPERTY TYPE'] == 'Studio', 'BEDROOMS'].fillna(0)

# Shared accommodation lets a single room
shares = ['House Share', 'Flat Share', 'House of Multiple Occupation']
df.loc[df['PROPERTY TYPE'].isin(shares), 'BEDROOMS'] = \
    df.loc[df['PROPERTY TYPE'].isin(shares), 'BEDROOMS'].fillna(1)

print("Still missing after imputation:", df['BEDROOMS'].isna().sum())

# Remaining missing values are genuinely unknown, not implied by property type
df = df[df['BEDROOMS'].notna()]
print("Shape after dropping remaining:", df.shape)

Still missing after imputation: 156
Shape after dropping remaining: (3203, 10)


### Postcode district coverage

The dataset includes a `subdistrict_code` column, but it is incomplete. The
addresses often contain the district, so the missing values can be recovered
by pattern matching rather than discarded.

In [9]:
#How many rows already have a distric code
print("subdistrict_code present :", df['subdistrict_code'].notna().sum(), "of", len(df))

# Proportion as a percentage: mean of a boolean column gives the fraction that are True
print("Coverage :", round(df['subdistrict_code'].notna().mean() * 100,1), "%")

# Inspect addresses with no code, to see whether the district is recoverable from the text
print("Examples of rows with missing codes:")
print(df[df['subdistrict_code'].isna()]['address'].head(10).tolist())

subdistrict_code present : 1736 of 3203
Coverage : 54.2 %
Examples of rows with missing codes:
['Rockingham Road, Uxbridge', 'Talbot Square', 'Phoenix Court Gasholder Place SE11', 'Gasholder Place London SE11', 'Falcon Wharf, Battersea Riverside', 'Xchange Point, Market Road, Islington, N7', 'Palace Court, London, W2', 'Lexham Gardens, London, W8', 'Union Wharf, London, N1', 'Hampden Road, London, N8']


### Recovering postcode districts from addresses

Where `subdistrict_code` is missing, most addresses still contain the district
in their text. A regular expression matching the UK postcode district format
recovers these, raising coverage without discarding listings. Addresses with no
district remain missing and are removed in the next step, since location is
required for both the model and the affordability benchmark.

In [10]:
# UK postcode district: 1-2 letters, 1-2 digits, optional trailing letter
# e.g. N7, SE11, W2, SW1V
pattern = r'\b([A-Z]{1,2}[0-9]{1,2}[A-Z]?)\b'

#Extract from address only where subdistrict_code is null
missing = df['subdistrict_code'].isna()
extracted = df.loc[missing, 'address'].str.extract(pattern, expand = False)
print("Recovered from address : ", extracted.notna().sum(), "of", missing.sum())

#Fill the missing cells in subdistrict_code with extracted codes from address
df.loc[missing, 'subdistrict_code'] = extracted

print("Coverage Now :", round(df['subdistrict_code'].notna().mean() * 100, 1), "%")

Recovered from address :  870 of 1467
Coverage Now : 81.4 %


### Validating extracted districts and present distrct codes

A regular expression matches a pattern, not a meaning, so it could capture text
resembling a postcode district without being one. Each extracted value is
therefore checked against the Urban Big Data Centre postcode lookup, which
contains every UK postcode. Values with no match are not genuine districts and
are removed.

In [11]:
# Validate extracted codes against real UK postcode districts
lookup = pd.read_csv('../data/raw/pcode_brma_lookup_v2.2.csv',
                     usecols=['PCDS'], dtype=str)

# PCDS holds a full postcode like "SW1A 1AA"; the district is the part before the space
real_districts = set(lookup['PCDS'].str.split().str[0].unique())
print("Real UK districts in lookup:", len(real_districts))

# Set difference: anything extracted that is not a genuine district
extracted_vals = set(df['subdistrict_code'].dropna().unique())
bogus = extracted_vals - real_districts
print("Not real postcode districts:", sorted(bogus) if bogus else "none")

Real UK districts in lookup: 3075
Not real postcode districts: ['Bow', 'E1W.', 'EC21', 'EC4', 'Iver', 'Kent', 'SW1', 'W2.', 'W8 6', 'WC2']


In [12]:
# Strip trailing punctuation and anything after a space
df['subdistrict_code'] = (df['subdistrict_code']
                          .str.strip()
                          .str.replace(r'[^A-Z0-9].*$', '', regex=True))

# Remove values that are not genuine districts
valid = df['subdistrict_code'].isin(real_districts)
print("Rows with an invalid district:", (~valid & df['subdistrict_code'].notna()).sum())

df.loc[~valid, 'subdistrict_code'] = pd.NA

print("Coverage after validation:", round(df['subdistrict_code'].notna().mean() * 100, 1), "%")

Rows with an invalid district: 10
Coverage after validation: 81.0 %


### Removing listings without a district

Location is required both as a model feature and to join each listing to its
affordability benchmark. Listings whose district could not be recovered from
either the original column or the address text are therefore removed.

In [13]:
# Location is required for both the model and the affordability benchmark
before = len(df)
df = df[df['subdistrict_code'].notna()]

print(f"Dropped {before - len(df)} rows with no recoverable district")
print("Shape:", df.shape)
print("Distinct districts:", df['subdistrict_code'].nunique())

Dropped 607 rows with no recoverable district
Shape: (2596, 10)
Distinct districts: 224


In [14]:
# rent is the prediction target — check it is numeric and sane
print(df['rent'].dtype)
print()
print(df['rent'].describe())
print()
print("Missing:", df['rent'].isna().sum())
print("Zero or negative:", (df['rent'] <= 0).sum())

int64

count     2596.000000
mean      2743.121341
std       2465.324511
min        449.000000
25%       1700.000000
50%       2400.000000
75%       3358.000000
max      52000.000000
Name: rent, dtype: float64

Missing: 0
Zero or negative: 0


In [15]:
# Inspect the highest rents to confirm they are real properties, not data errors
print(df.nlargest(10, 'rent')[['rent', 'subdistrict_code', 'PROPERTY TYPE', 'BEDROOMS']])

       rent subdistrict_code PROPERTY TYPE  BEDROOMS
147   52000              NW8      Detached       4.0
2849  52000               W8          Flat       5.0
3119  39000              SW7          Flat       3.0
3386  28167              SW7          Flat       3.0
954   28166              W1K          Flat       3.0
908   27083              SW3          Flat       3.0
172   23833             SW1X    Town House       3.0
2439  19933              W1H      Terraced       4.0
132   17333               W8          Flat       3.0
873   17117              NW8      Terraced       4.0


### Logarithmic transformation

The ten highest rents are located in NW8, W8, SW7, SW3, W1K and SW1X, which are
among the most expensive districts in London, and have three to five bedrooms.
These are genuine premium lettings rather than data errors. A logarithmic
transformation is therefore applied in preference to outlier removal, compressing
the right tail while retaining the full range of the market. The original rent
column is retained so that model error can be reported in pounds.

In [16]:
import numpy as np

# Compresses the right tail without discarding genuine premium lettings
df['rent_log'] = np.log(df['rent'])

print(df[['rent', 'rent_log']].describe())

               rent     rent_log
count   2596.000000  2596.000000
mean    2743.121341     7.729830
std     2465.324511     0.592668
min      449.000000     6.107023
25%     1700.000000     7.438384
50%     2400.000000     7.783224
75%     3358.000000     8.119101
max    52000.000000    10.858999


### Distinguishing shared from whole-property lettings

The dataset contains two distinct markets. A room in a shared house is priced on
a different basis from a whole property, and the inventory found medians of
roughly £800 against £2,600. A binary indicator allows the model to learn
separate price levels rather than treating them as a single population.

In [17]:
# Room lettings and whole-property lets are priced on different bases
share_types = ['House Share', 'Flat Share', 'House of Multiple Occupation']
df['is_shared'] = df['PROPERTY TYPE'].isin(share_types).astype(int)

print(df.groupby('is_shared')['rent'].agg(['count', 'median', 'mean']))

           count  median         mean
is_shared                            
0           2261  2600.0  3028.078284
1            335   802.0   819.874627


### Quality audit

The preceding steps addressed the issues documented in Section 3.4.1. The
remaining columns are examined before the dataset is finalised, to confirm no
further quality problems exist.

In [18]:
# Missing values across every remaining column
print("Missing values per column:")
print(df.isna().sum())
print()
print("Row count:", len(df))

Missing values per column:
address                              0
subdistrict_code                     0
rent                                 0
deposit                              0
Let type                           155
Furnish Type                         0
PROPERTY TYPE                       18
BEDROOMS                             0
BATHROOMS                          207
avg_distance_to_nearest_station      0
rent_log                             0
is_shared                            0
dtype: int64

Row count: 2596


In [19]:
# Columns not yet examined
for col in ['deposit', 'Let type', 'Furnish Type', 'BATHROOMS',
            'avg_distance_to_nearest_station']:
    print(f"--- {col} ---")
    print(f"dtype: {df[col].dtype}, missing: {df[col].isna().sum()}")
    if df[col].dtype == 'object':
        print(df[col].value_counts().head(6))
    else:
        print(df[col].describe())
    print()

--- deposit ---
dtype: int64, missing: 0
count     2596.000000
mean      2006.150616
std       2821.707973
min          0.000000
25%          0.000000
50%       1731.000000
75%       3000.000000
max      60000.000000
Name: deposit, dtype: float64

--- Let type ---
dtype: str, missing: 155
count          2441
unique            2
top       Long term
freq           2346
Name: Let type, dtype: object

--- Furnish Type ---
dtype: str, missing: 0
count          2596
unique            5
top       Furnished
freq           1569
Name: Furnish Type, dtype: object

--- BATHROOMS ---
dtype: float64, missing: 207
count    2389.000000
mean        1.457932
std         0.953048
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        20.000000
Name: BATHROOMS, dtype: float64

--- avg_distance_to_nearest_station ---
dtype: float64, missing: 0
count    2596.000000
mean        0.483243
std         0.288135
min         0.100000
25%         0.300000
50%         0.400000

In [20]:
print("--- Let type ---")
print(df['Let type'].value_counts(dropna=False))
print()
print("--- Furnish Type ---")
print(df['Furnish Type'].value_counts())
print()
print("--- deposit = 0 ---")
print("Count:", (df['deposit'] == 0).sum(), "of", len(df))
print()
print("--- PROPERTY TYPE missing: what are these rows? ---")
print(df[df['PROPERTY TYPE'].isna()][['rent', 'BEDROOMS', 'subdistrict_code']].head())
print()
print("--- BATHROOMS > 5 ---")
print(df[df['BATHROOMS'] > 5][['rent', 'BEDROOMS', 'BATHROOMS', 'subdistrict_code']])
print()
print("Exact duplicates:", df.duplicated().sum())
print("Duplicate address+rent:", df.duplicated(subset=['address', 'rent']).sum())

--- Let type ---
Let type
Long term     2346
NaN            155
Short term      95
Name: count, dtype: int64

--- Furnish Type ---
Furnish Type
Furnished                                         1569
Unfurnished                                        372
Furnished or unfurnished, landlord is flexible     271
Ask agent                                          220
Part furnished                                     164
Name: count, dtype: int64

--- deposit = 0 ---
Count: 838 of 2596

--- PROPERTY TYPE missing: what are these rows? ---
     rent  BEDROOMS subdistrict_code
217   875       1.0              HA9
553  2300       2.0              SW8
603  1950       2.0              TW3
628  3550       4.0               N1
633   850       1.0              UB8

--- BATHROOMS > 5 ---
       rent  BEDROOMS  BATHROOMS subdistrict_code
111    1500       1.0       20.0               N4
334   12500       6.0        6.0             WD25
557    1450       1.0       20.0               N4
720    1675      

### Removals arising from the audit

Short-term lets are priced per week or night and belong to a different market.
Listings with unknown property type cannot be classified as shared or whole.
Duplicates would leak between training and test data, inflating measured accuracy.

In [21]:
before = len(df)

# Short-term lets are priced on a different basis
df = df[df['Let type'] != 'Short term']
print(f"Short-term removed: {before - len(df)}")

# Property type unknown: cannot determine shared status
before = len(df)
df = df[df['PROPERTY TYPE'].notna()]
print(f"Unknown property type removed: {before - len(df)}")

# 20 bathrooms in a one-bedroom flat is a recording error
before = len(df)
df = df[(df['BATHROOMS'].isna()) | (df['BATHROOMS'] <= 10)]
print(f"Implausible bathroom counts removed: {before - len(df)}")

# Duplicate listings would leak between train and test splits
before = len(df)
df = df.drop_duplicates(subset=['address', 'rent', 'BEDROOMS'])
print(f"Duplicates removed: {before - len(df)}")

print("\nShape:", df.shape)

Short-term removed: 95
Unknown property type removed: 18
Implausible bathroom counts removed: 3
Duplicates removed: 41

Shape: (2439, 12)


In [22]:
share_types = ['House Share', 'Flat Share', 'House of Multiple Occupation']
df['is_shared'] = df['PROPERTY TYPE'].isin(share_types).astype(int)
print(df['is_shared'].value_counts())

is_shared
0    2113
1     326
Name: count, dtype: int64


### Scope mismatch between bedrooms and bathrooms

For shared lettings the bedroom field describes the single room being let while
the bathroom field describes the whole property. Left uncorrected, every
high-bathroom observation would be a low-priced room share, teaching the model
an inverse relationship that does not exist.

In [23]:
# Bathrooms should be plausible relative to bedrooms
ratio_check = df[df['BATHROOMS'].notna()].copy()
ratio_check['ratio'] = ratio_check['BATHROOMS'] / ratio_check['BEDROOMS'].replace(0, 1)

print("Listings with more than 2 bathrooms per bedroom:")
print(ratio_check[ratio_check['ratio'] > 2][
    ['rent', 'BEDROOMS', 'BATHROOMS', 'PROPERTY TYPE', 'is_shared']
].sort_values('BATHROOMS', ascending=False).head(20))
print()
print("Count:", (ratio_check['ratio'] > 2).sum())

Listings with more than 2 bathrooms per bedroom:
      rent  BEDROOMS  BATHROOMS PROPERTY TYPE  is_shared
2504   890       1.0        6.0   House Share          1
2507   890       1.0        6.0   House Share          1
2481   875       1.0        6.0   House Share          1
2386   845       1.0        6.0   House Share          1
2216   750       1.0        5.0   House Share          1
2309   800       1.0        5.0    Flat Share          1
2341   800       1.0        4.0   House Share          1
2160   695       1.0        4.0   House Share          1
1311  1075       1.0        4.0   House Share          1
2446   850       1.0        4.0   House Share          1
2440   850       1.0        4.0   House Share          1
2445   850       1.0        4.0   House Share          1
1616   850       1.0        4.0   House Share          1
2566   900       1.0        4.0   House Share          1
1112   950       1.0        3.0   House Share          1
2153   675       1.0        3.0   House

In [24]:
# For shared listings, BEDROOMS describes the let room but BATHROOMS describes
# the whole property. The two are measured at different scopes, so the bathroom
# count is not a property of the unit being priced.
print("Before:", df.loc[df['is_shared'] == 1, 'BATHROOMS'].notna().sum(), "shares have a bathroom count")

df.loc[df['is_shared'] == 1, 'BATHROOMS'] = pd.NA

print("After: ", df.loc[df['is_shared'] == 1, 'BATHROOMS'].notna().sum())
print("Total missing BATHROOMS now:", df['BATHROOMS'].isna().sum())

Before: 234 shares have a bathroom count
After:  0
Total missing BATHROOMS now: 429


In [25]:
print(df['subdistrict_code'].value_counts().describe())
print()
print("Districts with fewer than 5 listings:", (df['subdistrict_code'].value_counts() < 5).sum())

count    224.000000
mean      10.888393
std       27.127850
min        1.000000
25%        2.000000
50%        4.000000
75%        9.000000
max      227.000000
Name: count, dtype: float64

Districts with fewer than 5 listings: 123


In [26]:
print(df['subdistrict_code'].value_counts().head(10))

subdistrict_code
E14     227
W8      190
SW5     181
SW7     168
W14     101
E1       74
E16      62
SE1      52
SE16     39
E3       37
Name: count, dtype: int64


### Current state

The pipeline yields 2,439 listings across 224 postcode districts. The cleaned
output is not yet written to disk, pending confirmation of the training data
source.

In [27]:
# Final state of the cleaned dataset.
# Output is not written to disk pending confirmation of the data strategy.
print("Rows:", len(df))
print("Districts:", df['subdistrict_code'].nunique())
print("Columns:", df.columns.tolist())
print()
print("Missing values:")
print(df.isna().sum())

Rows: 2439
Districts: 224
Columns: ['address', 'subdistrict_code', 'rent', 'deposit', 'Let type', 'Furnish Type', 'PROPERTY TYPE', 'BEDROOMS', 'BATHROOMS', 'avg_distance_to_nearest_station', 'rent_log', 'is_shared']

Missing values:
address                              0
subdistrict_code                     0
rent                                 0
deposit                              0
Let type                           151
Furnish Type                         0
PROPERTY TYPE                        0
BEDROOMS                             0
BATHROOMS                          429
avg_distance_to_nearest_station      0
rent_log                             0
is_shared                            0
dtype: int64
